In [ ]:
# for colab setup
!nvidia-smi

In [ ]:
%%bash
# mini-infer Colab source sync and CUDA Toolkit check
set -e
if command -v nvcc >/dev/null 2>&1; then
  nvcc --version | tail -n 4
else
  echo "nvcc not found; nvidia-smi reports driver support, not the installed Toolkit"
fi
repo_dir="${MINI_INFER_DIR:-$PWD/mini-infer}"
if [ ! -d "$repo_dir/.git" ]; then
  git clone --depth 1 https://github.com/szfmsmdx/mini-infer.git "$repo_dir"
else
  if [ -n "$(git -C "$repo_dir" status --porcelain)" ]; then
    echo "Repository has local changes; resolve them before updating."
    exit 1
  fi
  git -C "$repo_dir" fetch origin main
  git -C "$repo_dir" switch --detach origin/main
fi
git -C "$repo_dir" rev-parse --short HEAD


In [ ]:
%%bash
# mini-infer Colab CUDA build: tests and benchmarks
set -eo pipefail
repo_dir="${MINI_INFER_DIR:-$PWD/mini-infer}"
build_dir="$repo_dir/build/cuda-colab"
cuda_arch="${CUDA_ARCH:-$(nvidia-smi -i 0 --query-gpu=compute_cap --format=csv,noheader | tr -d '.[:space:]')}"
cmake -S "$repo_dir" -B "$build_dir" -DCMAKE_BUILD_TYPE=Release -DBUILD_TESTING=ON -DBUILD_CUDA_BENCHMARKS=ON "-DCMAKE_CUDA_ARCHITECTURES=$cuda_arch"
cmake --build "$build_dir" -j 8


In [ ]:
%%bash
# mini-infer Colab tests (including GPU Tensor)
set -e
repo_dir="${MINI_INFER_DIR:-$PWD/mini-infer}"
ctest --test-dir "$repo_dir/build/cuda-colab" --output-on-failure


In [ ]:
%%bash
# CUDA perf: mini-infer vs llama.cpp ggml
set -e
repo_dir="${MINI_INFER_DIR:-$PWD/mini-infer}"
"$repo_dir/build/cuda-colab/tests/benchmarks/matmul_cuda_bench"
